# Test of the MVP_Analyzer V3

In [1]:

import sys
import os

sys.path.append(os.path.abspath(".."))
from PyMVP import Analyzer
import numpy as np
import matplotlib.pyplot as plt
import gsw


In [2]:
mvpa = Analyzer()

mvpa.load_mvp_data('/home/maxw/Documents/ESSTECH25/MVP300_DATA/Underway profile/Bbis to Abis - 6 - 6_5 Knots/')

# mvpa.load_mvp_data('/home/maxw/Documents/ESSTECH25/MVP300_DATA/Underway profile/A to B - 4 Knots/',delp=[8,9,10,11,18,19]) # manually delete profiles because the MVP was stopped and restarted during the cast


# mvpa.load_mvp_data_again('/home/maxw/Documents/ESSTECH25/MVP300_DATA/Underway profile/Abis to Bbis - 6 Knots/',delp=[6])
# mvpa.load_mvp_data_again('/home/maxw/Documents/ESSTECH25/MVP300_DATA/Underway profile/Bbis to Abis - 6 - 6_5 Knots/')
mvpa.load_ctd_data('/home/maxw/Documents/ESSTECH25/BATHYSONDE/DATA/TRAIT/NCDF/')


mvpa.keep_selected_profiles([0, 10, 8, 2], [0, 2, 4, 8])


Found 6 MVP files in the directory: /home/maxw/Documents/ESSTECH25/MVP300_DATA/Underway profile/Bbis to Abis - 6 - 6_5 Knots/
MVP data loaded successfully.
Found 5 CTD files in the directory: /home/maxw/Documents/ESSTECH25/BATHYSONDE/DATA/TRAIT/NCDF/
CTD data loaded successfully.


In [3]:
mvpa.mvp_correction(high_cutoff=0.2,dp=0.1)

Applying corrections to MVP profiles...


100%|██████████| 8/8 [00:02<00:00,  3.14it/s]

MVP profiles corrected.


In [4]:
mvpa.interpolate_CTD_and_MVPcorrected(10000)


CTD data interpolated onto corrected MVP pressure levels.


In [5]:
mvpa.print_profile_metadata()

MVP data:
Number of profiles: 4
  Profil down 0 - Profil up 1 - Latitude: 42.34332, Longitude: 6.09571, Date/Heure: 2025-08-10 11:44:25.400000
  Profil down 2 - Profil up 3 - Latitude: 42.77085, Longitude: 6.17328, Date/Heure: 2025-08-10 15:23:03.800000
  Profil down 4 - Profil up 5 - Latitude: 42.65474, Longitude: 6.15137, Date/Heure: 2025-08-10 14:27:35.800000
  Profil down 6 - Profil up 7 - Latitude: 42.40576, Longitude: 6.10565, Date/Heure: 2025-08-10 12:17:45.600000
CTD data:
Number of profiles: 4
  Profil down 0 - Profil up 1 - Latitude: 42.91386, Longitude: 6.21834, Date/Heure: 2025-08-09T08:27:35.000000000
  Profil down 2 - Profil up 3 - Latitude: 42.31252, Longitude: 6.09074, Date/Heure: 2025-08-10T22:40:15.000000000
  Profil down 4 - Profil up 5 - Latitude: 42.48060, Longitude: 6.12350, Date/Heure: 2025-08-11T02:04:24.000000000
  Profil down 6 - Profil up 7 - Latitude: 42.78910, Longitude: 6.17630, Date/Heure: 2025-08-11T08:42:36.000000000


In [6]:
from scipy.signal import savgol_filter, correlate
from scipy.interpolate import interp1d

def corrige_MVP_offset_on_ctd_simple(mvpa,id_mvp,id_ctd,min_depth):
    """
    This function corrects the offset between the MVP and CTD profiles by aligning the temperature, conductivity profiles. It calculates the mean difference in temperature between the two profiles and applies this correction to the CTD temperature data.
    id_mvp and id_ctd must be the same length as each MVp profile will be be corrected with the corresponding CTD profile. The function returns the corrected MVP temperature and conductivity profiles.
    """

    mean_temp_diff = []
    mean_cond_diff = []
    print("Calculating mean differences between MVP and CTD profiles before correction:")
    for i in range(len(id_mvp)):
        id_valid = mvpa.PRES_mvp_corr_interp[id_mvp[i]] >= min_depth
        # Calculate the mean difference in temperature between the MVP and CTD profiles
        temp_diff = np.nanmean(mvpa.TEMP_mvp_corr_interp[id_mvp[i], id_valid] - mvpa.TEMP_ctd_interp[id_ctd[i], id_valid])
        mean_temp_diff.append(temp_diff)

        cond_diff = np.nanmean(mvpa.COND_mvp_corr_interp[id_mvp[i], id_valid] - mvpa.COND_ctd_interp[id_ctd[i], id_valid])
        mean_cond_diff.append(cond_diff)
    print("Mean temperature difference between MVP and CTD profiles:", np.mean(mean_temp_diff))
    print("Mean conductivity difference between MVP and CTD profiles:", np.mean(mean_cond_diff))

    for i in range(len(id_mvp)):
        id_valid = mvpa.PRES_mvp_corr_interp[id_mvp[i]] >= min_depth

        # Calculate the mean difference in temperature between the MVP and CTD profiles
        temp_diff = np.nanmean(mvpa.TEMP_mvp_corr_interp[id_mvp[i], id_valid] - mvpa.TEMP_ctd_interp[id_ctd[i], id_valid])
        mvpa.TEMP_mvp_corr_interp[id_mvp[i]] -= temp_diff

        cond_diff = np.nanmean(mvpa.COND_mvp_corr_interp[id_mvp[i], id_valid] - mvpa.COND_ctd_interp[id_ctd[i], id_valid])
        mvpa.COND_mvp_corr_interp[id_mvp[i]] -= cond_diff


    mean_temp_diff = []
    mean_cond_diff = []
    print("After correction:")
    for i in range(len(id_mvp)):
        id_valid = mvpa.PRES_mvp_corr_interp[id_mvp[i]] >= min_depth

        # Calculate the mean difference in temperature between the MVP and CTD profiles
        temp_diff = np.nanmean(mvpa.TEMP_mvp_corr_interp[id_mvp[i], id_valid] - mvpa.TEMP_ctd_interp[id_ctd[i], id_valid])
        mean_temp_diff.append(temp_diff)

        cond_diff = np.nanmean(mvpa.COND_mvp_corr_interp[id_mvp[i], id_valid] - mvpa.COND_ctd_interp[id_ctd[i], id_valid])
        mean_cond_diff.append(cond_diff)
    print("Mean temperature difference between MVP and CTD profiles:", np.mean(mean_temp_diff))
    print("Mean conductivity difference between MVP and CTD profiles:", np.mean(mean_cond_diff))
   



def corrige_MVP_offset_on_ctd_exact(mvpa,id_mvp,id_ctd):
    """
    This function corrects the offset between the MVP and CTD profiles by aligning the temperature, conductivity profiles. It calculates the mean difference in temperature between the two profiles and applies this correction to the CTD temperature data.
    id_mvp and id_ctd must be the same length as each MVp profile will be be corrected with the corresponding CTD profile. The function returns the corrected MVP temperature and conductivity profiles.
    """

    mean_temp_diff = []
    mean_cond_diff = []

    print("Calculating mean differences between MVP and CTD profiles before correction:")
    for i in range(len(id_mvp)):
        # Calculate the mean difference in temperature between the MVP and CTD profiles
        temp_diff = np.nanmean(mvpa.TEMP_mvp_corr_interp[id_mvp[i]] - mvpa.TEMP_ctd_interp[id_ctd[i]])
        mean_temp_diff.append(temp_diff)

        cond_diff = np.nanmean(mvpa.COND_mvp_corr_interp[id_mvp[i]] - mvpa.COND_ctd_interp[id_ctd[i]])
        mean_cond_diff.append(cond_diff)
    print("Mean temperature difference between MVP and CTD profiles:", np.mean(mean_temp_diff))
    print("Mean conductivity difference between MVP and CTD profiles:", np.mean(mean_cond_diff))

    for i in range(len(id_mvp)):
        mvpa.TEMP_mvp_corr_interp[id_mvp[i]] = align_profiles(mvpa.PRES_mvp_corr_interp[id_mvp[i]], mvpa.TEMP_ctd_interp[id_ctd[i]], mvpa.TEMP_mvp_corr_interp[id_mvp[i]])[0]
        mvpa.COND_mvp_corr_interp[id_mvp[i]] = align_profiles(mvpa.PRES_mvp_corr_interp[id_mvp[i]], mvpa.COND_ctd_interp[id_ctd[i]], mvpa.COND_mvp_corr_interp[id_mvp[i]])[0]


    mean_temp_diff = []
    mean_cond_diff = []
    print("After correction:")
    for i in range(len(id_mvp)):
        # Calculate the mean difference in temperature between the MVP and CTD profiles
        temp_diff = np.nanmean(mvpa.TEMP_mvp_corr_interp[id_mvp[i]] - mvpa.TEMP_ctd_interp[id_ctd[i]])
        mean_temp_diff.append(temp_diff)

        cond_diff = np.nanmean(mvpa.COND_mvp_corr_interp[id_mvp[i]] - mvpa.COND_ctd_interp[id_ctd[i]])
        mean_cond_diff.append(cond_diff)
    print("Mean temperature difference between MVP and CTD profiles:", np.mean(mean_temp_diff))
    print("Mean conductivity difference between MVP and CTD profiles:", np.mean(mean_cond_diff))
   


def align_profiles(P, T_ref, T_to_align_raw, min_depth=0,max_shift=20):
    """
    Pipeline complet :
    - estime ΔP
    - recale
    - estime ΔT
    - corrige
    """

    ### 1. calcul delta de pression

    # Masque pour exclure les valeurs non finies
    mask_nan = (
    np.isfinite(P) &
    np.isfinite(T_ref) &
    np.isfinite(T_to_align_raw)
    )

    P = P[mask_nan]
    T_ref = T_ref[mask_nan]
    T_to_align = T_to_align_raw[mask_nan]

    # Masque pour exclure la surface
    # mask = P >= min_depth

    # P = P[mask]
    # T_ref = T_ref[mask]
    # T_to_align = T_to_align[mask]

    # Lissage léger
    T1s = savgol_filter(T_ref, 11, 2)
    T2s = savgol_filter(T_to_align, 11, 2)

    # Gradients
    dT1 = np.gradient(T1s, P)
    dT2 = np.gradient(T2s, P)

    # Normalisation (important pour corrélation)
    dT1 = (dT1 - np.mean(dT1)) / np.std(dT1)
    dT2 = (dT2 - np.mean(dT2)) / np.std(dT2)

    # Corrélation
    corr = correlate(dT2, dT1, mode='full')
    lags = np.arange(-len(dT1)+1, len(dT1))

    # Convertir en décalage en pression
    dP = np.mean(np.diff(P))
    shifts = lags * dP

    # Limiter les shifts plausibles
    valid = np.abs(shifts) <= max_shift

    deltaP = shifts[valid][np.argmax(corr[valid])]

    ### 2. recalage pression
    f = interp1d(P + deltaP, T_to_align, bounds_error=False, fill_value=np.nan)
    T_shifted = f(P)

    ### 3. calcul delta de température
    mask =  np.isfinite(T_ref) & np.isfinite(T_shifted)
    # mask = (P >= min_depth) & np.isfinite(T_ref) & np.isfinite(T_shifted)
    deltaT = np.median(T_shifted[mask] - T_ref[mask])

    ### 4. recalage thermique
    T_corrected = T_shifted - deltaT

    mask_corrected = np.isfinite(T_corrected)

    # copie pour ne pas modifier l'original directement
    T_out = T_to_align_raw.copy()

    # injection uniquement là où c’est valide
    T_out_indices = np.where(mask_nan)[0]
    T_out[T_out_indices[mask_corrected]] = T_corrected[mask_corrected]

    return T_out, deltaP, deltaT





In [7]:
corrige_MVP_offset_on_ctd_simple(mvpa,id_mvp=[0,1,2,3,4,5,6,7], id_ctd=[0,1,2,3,4,5,6,7],min_depth=200)

Calculating mean differences between MVP and CTD profiles before correction:
Mean temperature difference between MVP and CTD profiles: -0.04583440443096301
Mean conductivity difference between MVP and CTD profiles: -0.04246474466348184
After correction:
Mean temperature difference between MVP and CTD profiles: 8.559946369250123e-17
Mean conductivity difference between MVP and CTD profiles: 8.717693770608388e-16


In [9]:
%matplotlib tk
id = 4
plt.figure()
plt.plot(mvpa.TEMP_mvp_corr_interp[id], mvpa.PRES_mvp_corr_interp[id], label='MVP corrigé')
plt.plot(mvpa.TEMP_mvp_corr[id], mvpa.PRES_mvp_corr[id], label='MVP non recalé')
plt.plot(mvpa.TEMP_ctd_interp[id], mvpa.PRES_ctd_interp[id], label='CTD')
plt.gca().invert_yaxis()
plt.legend()